# DanceTrack – step 1: fine-tune the YOLOv8s detector on DanceTrack **train**
Run once. When training finishes the model is frozen as `outputs/dancetrack/detector_final.pt`, which the
main DanceTrack notebook picks up automatically.
Saved in outputs/dancetrack. If interrupted: set RESUME_TRAINING = True and re-run.

In [ ]:
# ── Cell 13: Fine-tune the detector on DanceTrack train (SELF-CONTAINED – can run right after a restart) ──
import os, sys, shutil, configparser
from pathlib import Path
import numpy as np, cv2
from tqdm.auto import tqdm
import torch
from ultralytics import YOLO

# ---------------- settings ----------------
DATA_ROOT    = Path(r"data/DanceTrack")
WORK_DIR     = Path(r"outputs/dancetrack")
BASE_MODEL   = 'yolov8s.pt'
EPOCHS       = 10   # DanceTrack: 10 epochs (as in the paper)
TRAIN_IMGSZ  = 1280          # use 640 if you must train on CPU
BATCH        = 8             # lower to 4 if "CUDA out of memory"
FRAME_STRIDE = 10            # every 10th train frame (DanceTrack videos are long)
RESUME_TRAINING = False      # True = continue an interrupted run from last.pt (same device/settings)
RUN_FINETUNE = True          # !! set True ONLY to (re)train the detector – it overwrites best.pt !!

if not RUN_FINETUNE:
    print('Cell 13 skipped (RUN_FINETUNE = True ). Your detector is unchanged. For the test submission run Cell 14.')
else:
    # ---------------- device check ----------------
    print('Python :', sys.executable)
    print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
    DEVICE = 0 if torch.cuda.is_available() else 'cpu'
    if DEVICE == 'cpu':
        print('WARNING: training on CPU will be very slow – install the CUDA build of PyTorch first.')
    else:
        print('GPU    :', torch.cuda.get_device_name(0))

    # ---------------- helpers ----------------
    IMG_EXTS = {'.jpg', '.jpeg', '.png'}
    def is_seq(d): return d.is_dir() and (d / 'img1').is_dir()
    def find_split(root, split):
        for c in [root / split, root / 'dataset' / split, root / 'sportsmot_publish' / 'dataset' / split,
                  root / 'dancetrack' / split]:
            if c.is_dir() and any(is_seq(d) for d in c.iterdir()): return c
        raise FileNotFoundError(f'"{split}" split not found under {root}')

    def export(split_dir, part, stride, yroot):
        (yroot / 'images' / part).mkdir(parents=True, exist_ok=True)
        (yroot / 'labels' / part).mkdir(parents=True, exist_ok=True)
        n = 0
        for s in tqdm(sorted(d for d in split_dir.iterdir() if is_seq(d)), desc=f'export {part}'):
            gt_file = s / 'gt' / 'gt.txt'
            if not gt_file.exists(): continue
            gt = np.loadtxt(gt_file, delimiter=',', ndmin=2)
            if gt.shape[1] >= 7: gt = gt[gt[:, 6] != 0]
            cp = configparser.ConfigParser(); cp.read(s / 'seqinfo.ini')
            W, H = int(cp['Sequence']['imWidth']), int(cp['Sequence']['imHeight'])
            frames = sorted(p for p in (s / 'img1').iterdir() if p.suffix.lower() in IMG_EXTS)
            for p in frames[::stride]:
                fno = int(p.stem)
                dst = yroot / 'images' / part / f'{s.name}_{fno:06d}{p.suffix}'
                if not dst.exists(): shutil.copy(p, dst)
                lab = []
                for row in gt[gt[:, 0] == fno]:
                    x, y, w, h = row[2:6]
                    x1, y1, x2, y2 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)
                    if x2 - x1 > 1 and y2 - y1 > 1:
                        cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
                        bw, bh = (x2 - x1) / W, (y2 - y1) / H
                        lab.append(f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
                (yroot / 'labels' / part / f'{dst.stem}.txt').write_text(chr(10).join(lab))
                n += 1
        return n

    # ---------------- dataset (skipped if already exported) ----------------
    yroot = WORK_DIR / 'yolo_sportsmot'
    if not (yroot / 'data.yaml').exists():
        print('train images:', export(find_split(DATA_ROOT, 'train'), 'train', FRAME_STRIDE, yroot))
        print('val images  :', export(find_split(DATA_ROOT, 'val'), 'val', 10 * FRAME_STRIDE, yroot))
        yaml_lines = ['path: ' + yroot.as_posix(),
                      'train: images/train',
                      'val: images/val',
                      'names:',
                      '  0: player']
        (yroot / 'data.yaml').write_text(chr(10).join(yaml_lines) + chr(10))
    else:
        print('Dataset already exported:', yroot)

    # ---------------- train ----------------
    run_dir = WORK_DIR / 'yolo_runs' / 'sportsmot_player'
    last = run_dir / 'weights' / 'last.pt'
    if RESUME_TRAINING and last.exists():
        print('Resuming from', last)
        res = YOLO(str(last)).train(resume=True)
    else:
        res = YOLO(BASE_MODEL).train(
            data=str(yroot / 'data.yaml'), epochs=EPOCHS, imgsz=TRAIN_IMGSZ,
            batch=BATCH, device=DEVICE, project=str(WORK_DIR / 'yolo_runs'),
            name='sportsmot_player', exist_ok=True, workers=0, patience=10)
    best = run_dir / 'weights' / 'best.pt'
    print('\nFine-tuned weights:', best, '| exists:', best.exists())
    # Freeze the finished model as detector_final.pt (read-only, never overwritten)
    import stat
    final_det = WORK_DIR / 'detector_final.pt'
    if last.exists() and not final_det.exists():
        ck = torch.load(last, map_location='cpu', weights_only=False)
        if ck.get('optimizer') is None:                 # optimizer is stripped only when training finished
            shutil.copy2(last, final_det); os.chmod(final_det, stat.S_IREAD)
            print('Saved final detector (read-only):', final_det)
        else:
            print('Training not finished – detector_final.pt NOT created yet.')
    elif final_det.exists():
        print('detector_final.pt already exists – not changed:', final_det)
    print('Cell 2 now picks detector_final.pt automatically – just Run All.')